In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gzip
import shutil
import os
from src.utils.ss_dataset import *
import json
import time
import sys
import traceback

### Some Utilities ###

In [3]:
def get_deep_size(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)
    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(get_deep_size(k, seen) + get_deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set)):
        size += sum(get_deep_size(item, seen) for item in obj)

# deep search in dict
def find_parent_key(data, target_key:str, parent_key=None):
    if isinstance(data, dict):
        if target_key in data:
            return parent_key
        for k, v in data.items():
            result = find_parent_key(v, target_key, k)
            if result is not False:
                return result
    elif isinstance(data, list):
        for item in data:
            result = find_parent_key(item, target_key, parent_key)
            if result is not False:
                return result
    return False


# string search in raw_text
def find_substring_span(text: str, search: str) -> tuple[int, int] | None:
    start = text.find(search)
    if start == -1:
        return None
    end = start + len(search)
    return start, end

def get_abstract_of_paper(paper_id:str,data_bank:dict):
    try:
        paper_data = data_bank[paper_id]["data"]
        parent_annotations = find_parent_key(paper_data,target_key="annotations")
        parent_text = find_parent_key(paper_data,"text")
        annotation_data = paper_data[parent_annotations]["annotations"]
        raw_text = paper_data[parent_text]["text"]
        status,info = extract_info_w_attributes(raw_text,annotation_data,"abstract")
    except Exception as e:
        print(traceback.format_exc())
        status = False
        info = []
    return status,info

In [4]:
FINAL_DATASET = "/store/psm-citation-ranking/dataset/processed/s2orc-citation-ranker-240626/final_dataset.jsonl"

**Load Final dataset - this can consume large memory**

In [5]:
citation_data = {}
with open(FINAL_DATASET, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        corpus_id = str(data['corpusid'])
        paper_type = data["paper_type"]
        if corpus_id not in citation_data.keys():
            citation_data[corpus_id] = {"data":data,"paper_type":[paper_type]}
        else:
            citation_data[corpus_id]["paper_type"].append(paper_type)
print(f"Total papers {len(citation_data)}")

# citation_data_bytes = get_deep_size(citation_data)
# print(f"citation_data memory usage: {citation_data_bytes / (1024 ** 2):.2f} MB")


Total papers 21789


**Check how many Main papers**

In [6]:
main_papers = []
for id_ in citation_data.keys():
    paper_type = citation_data[id_]['paper_type']
    if "main" in paper_type:
        main_papers.append(id_) # just collecting the ids of main papers
nb_main = len(main_papers)
print(f"Total Main {nb_main} papers")

Total Main 2698 papers


### **At random pick one paper and explore the citations and citation details** ###

In [15]:
import random
index = random.randint(0, nb_main-1)  # inclusive of both 0 and N-1
main_corpusid = main_papers[index]
# main_corpusid = '257892508'
print(f"random Index {index}- Paper ID {main_corpusid}")
bib_entries = eval(citation_data[main_corpusid]["data"]['content']['annotations']['bibentry'])
for b_ent in bib_entries:
    try:
        matched_paper_id = str(b_ent['attributes']['matched_paper_id'])
        success = True if matched_paper_id in citation_data.keys() else False
        print(f"{main_corpusid}-->{matched_paper_id} [{success}]")
    except KeyError:
        print(f"matched Paper Not found for entry {b_ent['attributes']}")

random Index 2493- Paper ID 231761220
231761220-->219724368 [True]
231761220-->210708956 [True]
231761220-->227038123 [True]
231761220-->221238510 [True]
231761220-->221235836 [True]
231761220-->218983321 [True]
231761220-->218534824 [True]
231761220-->217591186 [True]
231761220-->215792438 [True]
231761220-->214680636 [True]
231761220-->211525155 [True]
231761220-->211101736 [True]
231761220-->209513737 [True]
231761220-->211025560 [True]
231761220-->204740427 [True]


In [16]:
citation_data[main_corpusid] # this will show the schema

{'data': {'corpusid': 231761220,
  'externalids': {'arxiv': None,
   'mag': None,
   'acl': None,
   'pubmed': '33525636',
   'pubmedcentral': '7865375',
   'dblp': 'journals/sensors/PiconeCV21',
   'doi': '10.3390/s21030892'},
  'content': {'source': {'pdfurls': None,
    'pdfsha': 'e8bb24ba8140d62a7934b504beba94a52ce27344',
    'oainfo': {'license': 'CCBY',
     'openaccessurl': 'https://doi.org/10.3390/s21030892',
     'status': 'GOLD'}},
   'text': "\nBlockchain Security and Privacy for the Internet of Things\n\n\nMarco Picone \nDepartment of Sciences and Methods for Engineering (DISMI)\nUniversity of Modena and Reggio Emilia\nVia Amendola 2, Pad. Morselli42121Reggio EmiliaItaly\n\nSimone Cirani \nDirector of Engineering\nTaneto di Gattatico (RE)\nCaligoo srl, Via Don Minzoni112, 43024Italy\n\nDepartment of Engineering and Architecture\nUniversity of Parma\nParco Area delle Scienze, 181/A43124ParmaItaly\n\nLuca Veltri \nDepartment of Engineering and Architecture\nUniversity of Parm

In [9]:
paper_data = citation_data[main_corpusid]["data"]
parent_annotations = find_parent_key(paper_data,target_key="annotations")
parent_text = find_parent_key(paper_data,"text")
annotation_data = paper_data[parent_annotations]["annotations"]
raw_text = paper_data[parent_text]["text"]


**what are the available annotations ?**

In [10]:
attribute_names = list(annotation_data.keys())
for name in attribute_names:
    info = annotation_data[name]
    if info is not None:
        print(f"{name}\n{info}\n------\n")

abstract
[{"end":912,"start":78},{"end":912,"start":78}]
------

bibauthorfirstname
[{"end":22706,"start":22705},{"end":22718,"start":22715},{"end":22727,"start":22726},{"end":22729,"start":22728},{"end":22740,"start":22739},{"end":22748,"start":22743},{"end":22757,"start":22756},{"end":22768,"start":22767},{"end":22770,"start":22769},{"end":22782,"start":22781},{"end":22784,"start":22783},{"end":23213,"start":23212},{"end":23224,"start":23223},{"end":23237,"start":23236},{"end":23239,"start":23238},{"end":23586,"start":23585},{"end":23597,"start":23596},{"end":23599,"start":23598},{"end":22706,"start":22705},{"end":22718,"start":22715},{"end":22727,"start":22726},{"end":22729,"start":22728},{"end":22740,"start":22739},{"end":22748,"start":22743},{"end":22757,"start":22756},{"end":22768,"start":22767},{"end":22770,"start":22769},{"end":22782,"start":22781},{"end":22784,"start":22783},{"end":23213,"start":23212},{"end":23224,"start":23223},{"end":23237,"start":23236},{"end":23239,"start

**Check here some annotations that are not None**

In [14]:
field_name = "title"
status,info = extract_info_w_attributes(raw_text,annotation_data,field_name)
for d in info:
    print(d[-1])

Web Appendix A1 Sample software code
Web Appendix A1 Sample software code
Web Appendix A1 Sample software code
Web Appendix A1 Sample software code


**View Citations and dig deeper**

In [12]:
# get Citations
status,bibref = extract_info_w_attributes(raw_text,annotation_data,"bibref") # these are refs like [1],[2] etc
if status:
    all_bibrefs = process_bibref(bibref) # this is a dict like {"b0":[1]...}
    status,bibentry = extract_info_w_attributes(raw_text,annotation_data,"bibentry") # these are entries like paper details,author etc
    raw_citations = len(bibentry)
    all_bibentries,nb_citations = process_bibentries(bibentry,all_bibrefs)
    for ent in all_bibentries:
        cite_key = all_bibentries[ent]["cite_key"]
        cite_text = all_bibentries[ent]["cite_text"]
        citation_id = all_bibentries[ent]["matched_paper_id"] # this paper id you have
        # knowing more about citations
        found = find_substring_span(raw_text,cite_text)
        st,abstract = get_abstract_of_paper(str(citation_id),citation_data)
        print(cite_key) # {"cite_key":[3],"cite_text":This is cited as....,"matched_paper_id":one_corpusid}
        print(cite_text)
        if found is not None:
            print(f"Found in {found[0]}-{found[1]}")
        print(citation_id)
        if st:
            citation_abstract = abstract[0][-1]
            print(citation_abstract)
        print("--------------")

1
Assessing the suitability of summary data for two-sample Mendelian randomization analyses using MR-Egger regression: the role of the I2 statistic. J Bowden, Del Greco, M F Minelli, C , Davey Smith, G Sheehan, N A Thompson, J R , Int J Epidemiol. 456Bowden J, Del Greco M. F, Minelli C, Davey Smith G, Sheehan NA, Thompson JR. Assessing the suitability of summary data for two-sample Mendelian randomization analyses using MR-Egger regression: the role of the I2 statistic. Int J Epidemiol. 2016;45(6):1961-1974.
Found in 22558-23069
3822103
--------------
2,
Combining information on multiple instrumental variables in Mendelian randomization: comparison of allele score and summarized data methods. S Burgess, F Dudbridge, S G Thompson, Stat Med. 3511Burgess S, Dudbridge F, Thompson SG. Combining information on multiple in- strumental variables in Mendelian randomization: comparison of allele score and summarized data methods. Stat Med. 2016;35(11):1880-1906.
Found in 23071-23476
9007790
----

**something wrong ? check specific paper**

In [13]:
paper_id = "3822103"
annotation_data = citation_data[main_corpusid]["data"]["content"]["annotations"]
raw_text = citation_data[main_corpusid]["data"]["content"]["text"]
print(annotation_data)
get_abstract_of_paper(paper_id,citation_data)

{'abstract': '[{"end":912,"start":78},{"end":912,"start":78}]', 'author': None, 'authoraffiliation': None, 'authorfirstname': None, 'authorlastname': None, 'bibauthor': None, 'bibauthorfirstname': '[{"end":22706,"start":22705},{"end":22718,"start":22715},{"end":22727,"start":22726},{"end":22729,"start":22728},{"end":22740,"start":22739},{"end":22748,"start":22743},{"end":22757,"start":22756},{"end":22768,"start":22767},{"end":22770,"start":22769},{"end":22782,"start":22781},{"end":22784,"start":22783},{"end":23213,"start":23212},{"end":23224,"start":23223},{"end":23237,"start":23236},{"end":23239,"start":23238},{"end":23586,"start":23585},{"end":23597,"start":23596},{"end":23599,"start":23598},{"end":22706,"start":22705},{"end":22718,"start":22715},{"end":22727,"start":22726},{"end":22729,"start":22728},{"end":22740,"start":22739},{"end":22748,"start":22743},{"end":22757,"start":22756},{"end":22768,"start":22767},{"end":22770,"start":22769},{"end":22782,"start":22781},{"end":22784,"sta

(False, [])